In [201]:
%pip install dagshub mlflow

In [1]:
import dagshub
import mlflow

In [2]:
dagshub.init(repo_owner='leosh1d', repo_name='gp5', mlflow=True)

Accessing as leosh1d

Initialized MLflow to track repo "leosh1d/gp5"

Repository leosh1d/gp5 initialized!

In [3]:
mlflow.set_tracking_uri("https://dagshub.com/leosh1d/gp5.mlflow/")
mlflow.set_experiment("CNN exp")

<Experiment: artifact_location='mlflow-artifacts:/701a8bb0ac77456aa60179b4f583659e', creation_time=1781456802311, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1781456802311, lifecycle_stage='active', name='CNN exp', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

Логирование первично подключили, пойдем дальше файлами

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os
import shutil
INPUT_DIR = "/content/drive/MyDrive/food11"
train_dir = os.path.join(INPUT_DIR, "train")
val_dir = os.path.join(INPUT_DIR, "val")

In [6]:
import json
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from torchvision import transforms

In [7]:
#код взят с семинара
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])
DEVICE=(torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
BATCH_SIZE = 128

In [8]:
#код взят с семинара
data_transforms = {
    "train": transforms.Compose([
            transforms.Resize(256) ,#сначала меняем размер с запасом
            transforms.CenterCrop(244),
            transforms.RandomHorizontalFlip(), #зеркальное отражение по горизонтали
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD), #нормализация
    ]),
    "val": transforms.Compose([# На валидации и тесте только изменение размера и кроп по центру
            transforms.Resize(256),
            transforms.CenterCrop(244),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    "test": transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(254),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
}

In [9]:
#код взят с семинара
dataset = {ds: torchvision.datasets.ImageFolder(
        root=os.path.join(INPUT_DIR, ds),
        transform=data_transforms[ds]
    ) for ds in ["train","val","test"]}

dataset_size={ds:len(dataset[ds]) for ds in ["train","val","test"]}
dataset_classes = dataset["train"].classes
print("classes:", dataset_classes, "\nsize", dataset_size)

classes: ['apple_pie', 'cheesecake', 'chicken_curry', 'french_fries', 'fried_rice', 'hamburger', 'hot_dog', 'ice_cream', 'omelette', 'pizza', 'sushi'] 
size {'train': 8828, 'val': 1101, 'test': 1100}


In [26]:
#код взят с семинара
dataloader = {
  "train": torch.utils.data.DataLoader(
      dataset=dataset["train"], batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
  ),
  "val": torch.utils.data.DataLoader(
      dataset=dataset["val"],batch_size=BATCH_SIZE,shuffle=False,num_workers=2,
  ),
  "test": torch.utils.data.DataLoader(
      dataset=dataset["test"],batch_size=BATCH_SIZE,shuffle=False,num_workers=2,
  ),
}

In [11]:
import os
import time
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm

In [12]:
def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train()
    total = 0
    correct = 0
    total_loss = 0

    n_ex = len(train_loader)

    for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=n_ex):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad() # обнуляем градиенты
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)
        pred = output.argmax(dim=1)
        correct += (pred == target).sum().item() # скока предсказаний совпало с правильными ответами
        train_loss = criterion(output, target) # считаем лосс
        train_loss.backward() # обратный проход
        optimizer.step() # делаем шаг оптимизатором

        total_loss += train_loss.item() * data.size(0)
        total += target.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    tqdm.write(
    f'Epoch {epoch}: Average loss={avg_loss:.4f}, Accuracy={accuracy * 100:.2f}%')

    return avg_loss, accuracy

In [13]:
def test(model, device, test_loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target)
            total_loss += test_loss.item() * data.size(0)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)
    avg_loss = total_loss / total
    accuracy = correct / total

    tqdm.write(
        f'Test set: Average loss={avg_loss:.4f}, Accuracy={accuracy * 100:.2f}%')

    return avg_loss, accuracy

In [27]:
NUM_EPOCHS=10
LR =0.003
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def launch(model,model_name,train_loader,val_loader,test_loader,num_epochs):

    model = model().to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    # отправляем данные о нпачале прикола
    with mlflow.start_run(run_name=model_name) as run:
        mlflow.log_params({
            "model_name": model_name,
            "num_epochs": num_epochs,
            "lr": LR,
            "batch_size": BATCH_SIZE,
            "num_params": sum(p.numel() for p in model.parameters()),
        })


        for epoch in range(1, num_epochs + 1):
            print('Epoch:', epoch)

            train_loss, train_acc = train(model,device,train_loader,optimizer,criterion,epoch)
            val_loss, val_acc = test(model,device,val_loader,criterion)

            # лог внутри эпохи
            mlflow.log_metrics({
                "train_loss": train_loss, "train_acc": train_acc,
                "val_loss": val_loss, "val_acc": val_acc,
            }, step=epoch)


        # оттренили модель, теперь тестируем ее
        test_loss, test_acc = test(model,device,test_loader,criterion)

        test_metrics = {
            "test_loss": test_loss,
            "test_acc": test_acc,
        }

        mlflow.log_metrics(test_metrics)
        mlflow.pytorch.log_model(model, artifact_path="model", serialization_format='pt2')


    print('победили!')

Данных очень много, обучение долгое. Начнем с очень простой модели:

In [15]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=11):
        super().__init__()
        self.model = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1), # 244 -> 122
        nn.ReLU(),

        # Блок 2: 122 -> 61
        nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
        nn.ReLU(),

        # Блок 3: 61 -> 31
        nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
        nn.ReLU(),

        # Блок 4: 31 -> 16
        nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
        nn.ReLU(),

        # Блок 5: 16 -> 8
        nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1),
        nn.ReLU(),

        # Блок 6: 8 -> 4  (новый блок для 244)
        nn.Conv2d(512, 512, kernel_size=3, stride=2, padding=1),
        nn.ReLU(),

        nn.AdaptiveAvgPool2d(1),

        nn.Flatten(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.model(x)


In [20]:
launch(
    model=SimpleCNN,
    model_name="SimpleCNN",
    train_loader=dataloader['train'],
    val_loader=dataloader['val'],
    test_loader=dataloader['test'],
    num_epochs=NUM_EPOCHS,
)

KeyboardInterrupt: 

Мда ну дела вообще плохи, но тут очень упрощенная очень быстрая модель. Навесим на нее улучшалок

In [218]:
class ImprovedCNN(nn.Module):
    def __init__(self, num_classes=11):
        super().__init__()
        self.model = nn.Sequential(
            # Блок 1: 160 -> 80
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=0.1),

            # Блок 2: 122 -> 61
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=0.1),

            # Блок 3: 61 -> 30
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=0.2),

            # Блок 4: 30 -> 15
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=0.2),

            # Блок 5: 15 -> 7
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=0.3),

            # Блок 6: 7 -> 3
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=0.3),

            nn.AdaptiveAvgPool2d(1),

            nn.Flatten(),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(p=0.5),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.model(x)


In [219]:
launch(
    model=ImprovedCNN,
    model_name="ImprovedCNN",
    train_loader=dataloader['train'],
    val_loader=dataloader['val'],
    test_loader=dataloader['test'],
    num_epochs=NUM_EPOCHS,
)

Epoch: 1


100%|██████████| 69/69 [01:16<00:00,  1.11s/it]

Epoch 1: Average loss=2.3033, Accuracy=18.87%


Test set: Average loss=2.1270, Accuracy=26.70%
Epoch: 2


100%|██████████| 69/69 [01:17<00:00,  1.12s/it]

...

100%|██████████| 69/69 [01:16<00:00,  1.10s/it]

Epoch 10: Average loss=1.7100, Accuracy=42.94%


Test set: Average loss=1.5392, Accuracy=49.14%
Test set: Average loss=1.6410, Accuracy=45.00%


2026/06/16 01:36:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 01:36:44 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/16 01:36:44 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/16 01:36:51 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label

🏃 View run ImprovedCNN at: https://dagshub.com/leosh1d/gp5.mlflow/#/experiments/0/runs/119cf22f255844109dfbe6de82e2d23f
🧪 View experiment at: https://dagshub.com/leosh1d/gp5.mlflow/#/experiments/0
победили!


Мы добавили batchNorm и dropout. Первый дал нам ускорение и немного регуляризации (чтобы не переобучиться), а второй не дал нам зависеть от конкретных нейронов путем их дропа. Это тоже нам правит переобучение. Результат на лицо, но еще поработаем)

Теперь придем к самому жиру, сделаем ResNet, чтобы ошибка проходила между слоями дальше.

In [16]:
class ResBlock(nn.Module):

    def __init__(self, channels, dropout_p=0.1):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Dropout2d(p=dropout_p),

            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )

        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.block(x) + x)  # это дает скип соединения


In [17]:
class DownBlock(nn.Module):

    def __init__(self, in_channels, out_channels, dropout_p=0.1):
        super().__init__()

        # Основной путь
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Dropout2d(p=dropout_p),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
        )

        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1),
            nn.BatchNorm2d(out_channels),
        )

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.relu(self.block(x) + self.shortcut(x))  # Тоже скип соединения
        return self.pool(out)


In [18]:
class ResNet(nn.Module):

    def __init__(self, num_classes=11):
        super().__init__()

        self.network = nn.Sequential(

            # Блок 1: 244 -> 122 | 3 -> 32
            DownBlock(3, 32, dropout_p=0.1),
            ResBlock(32, dropout_p=0.1),

            # Блок 2: 122 -> 61 | 32 -> 64
            DownBlock(32, 64, dropout_p=0.1),
            ResBlock(64, dropout_p=0.1),

            # Блок 3: 61 -> 30 | 64 -> 128
            DownBlock(64, 128, dropout_p=0.2),
            ResBlock(128, dropout_p=0.2),

            # Блок 4: 30 -> 15 | 128 -> 256
            DownBlock(128, 256, dropout_p=0.2),
            ResBlock(256, dropout_p=0.2),

            # Блок 5: 15 -> 7 | 256 -> 512
            DownBlock(256, 512, dropout_p=0.3),
            ResBlock(512, dropout_p=0.3),

            # Блок 6: 7 -> 3 | 512 -> 512
            DownBlock(512, 512, dropout_p=0.3),
            ResBlock(512, dropout_p=0.3),

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )

        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(p=0.5),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.network(x)
        return self.classifier(x)

тут я поймал ошибку с вываливанием за память ГПУ, пошел рестартить рантайм

In [34]:
import gc; gc.collect()
import torch; torch.cuda.empty_cache()

In [25]:
BATCH_SIZE = 32

In [ ]:
launch(
    model=ResNet,
    model_name="ResNet",
    train_loader=dataloader['train'],
    val_loader=dataloader['val'],
    test_loader=dataloader['test'],
    num_epochs=NUM_EPOCHS,
)

Epoch: 1


100%|██████████| 276/276 [01:54<00:00,  2.41it/s]

Epoch 1: Average loss=2.4039, Accuracy=14.18%


Test set: Average loss=2.2655, Accuracy=16.08%
Epoch: 2


100%|██████████| 276/276 [01:55<00:00,  2.38it/s]

...

Test set: Average loss=2.0358, Accuracy=31.34%
Epoch: 5


100%|██████████| 276/276 [01:55<00:00,  2.38it/s]

Epoch 5: Average loss=1.8729, Accuracy=36.09%


Test set: Average loss=1.8490, Accuracy=36.88%
Epoch: 6


 76%|███████▌  | 210/276 [01:29<00:28,  2.34it/s]